# Jouyit Matthieu

# AI Usage

I used AI to implement and test the various SNARIMAX configurations.
It also helped me match the colors and themes of Streamlit with the plots. It also helped me with the GeoJSON files and with implementing forecasting in Streamlit (with or without weather data).

# Description

### Forecasting function – Model selection and experiments

Before implementing the final forecasting function, several experiments were conducted using the River framework and its SNARIMAX model. The objective was to identify suitable model parameters, determine the appropriate length of historical data and evaluate the impact of the forecast horizon.

All experiments were initially performed on the Norwegian electricity region **EL1 (NO1)**, using weekly reservoir filling data from Magasinstatistikk.

**1. Selection of the data source and time resolution**

Only Magasinstatistikk was retained as the input data source. Meteorological variables from Open-Meteo were considered, but the available weather dataset covered only one year (2024), which was insufficient to reliably explain long-term reservoir filling patterns. The reservoir data was therefore kept at a weekly frequency, providing a suitable balance between temporal detail and computational efficiency.

**2. Model evaluation and parameter selection**

Several SNARIMAX configurations were tested and compared using historical test periods from 2023, 2024 and 2025. Different combinations of non-seasonal and seasonal parameters were evaluated.

The configuration `(p=1, d=1, q=1, sp=1, sd=1, sq=0, m=52)` provided the best overall compromise among the tested SNARIMAX configurations. The seasonal period of 52 corresponds to the annual seasonality of weekly observations.

The model was also compared against a Seasonal Naive baseline, which predicts future values by repeating the values observed during the corresponding weeks of the previous year. This baseline performed competitively, highlighting the difficulty of long-term reservoir filling forecasts.

**3. Selection of the historical period**

Different historical window lengths were tested: 2, 3, 5, 7 and 10 years. The results showed that longer histories could improve the average prediction error, although the best period varied depending on the test year. Three years was retained as a reasonable default, offering a good compromise between recent information and sufficient historical observations.

**4. Evaluation of forecast horizons**

The model was evaluated over four forecast horizons: 4, 13, 26 and 52 weeks, corresponding approximately to 1, 3, 6 and 12 months.

As expected, prediction accuracy generally decreased as the forecast horizon increased. The 13-week horizon was selected as the default because it provides a useful medium-term forecast while maintaining relatively good performance.

**5. Final implementation**

Based on these experiments, a reusable function called `forecast_reservoir_filling()` was developed. It allows the user to select the region, historical period, forecast horizon and SNARIMAX parameters. The default configuration uses EL1, three years of historical data and a 13-week forecast horizon.

The function returns a DataFrame containing the predicted reservoir filling degrees and their corresponding future dates. A Plotly visualization was also implemented to compare historical observations with future predictions.

### Streamlit application – Structure and navigation

The application is organized into four main sections, each corresponding to a specific type of task performed by the user.

**1. Home**

The Home page provides an overview of the application, introduces its main functionalities and gives users access to the different sections.

**2. Data Exploration**

- **Reservoir measurements:** exploration of historical Norwegian reservoir measurements, including filling levels and monthly summaries.
- **Cross-border flows:** exploration of physical electricity exchanges between Norway and neighbouring European bidding zones.
- **Geographical map:** geographical visualization of Norwegian reservoir locations and related information.

**3. Data Analysis**

- **Reservoir trends:** visualization and analysis of reservoir indicators over selected time periods.
- **Reservoir analysis:** investigation of seasonal and frequency characteristics using STL decomposition and spectrograms.

**4. Advanced Analytics**

- **Weather anomalies:** detection of unusual temperature and precipitation observations using Local Outlier Factor (LOF) and Statistical Process Control (SPC).
- **Meteorology & Reservoir Correlation:** investigation of relationships between meteorological variables, particularly temperature and precipitation, and reservoir filling levels.
- **Reservoir Forecasting:** prediction of future reservoir filling levels using the SNARIMAX model, with the possibility of incorporating temperature and precipitation as exogenous variables.

This structure separates data exploration, analytical methods and advanced predictive functionalities. It provides a clear navigation experience while making it easier to integrate additional analytical and machine learning methods in the future.

### Meteorology and reservoir correlation - findings

For NO1, I investigated the relationship between temperature and reservoir filling degree using different time lags.

The Pearson correlation increased progressively:

- Lag 0: 0.438
- Lag 1: 0.533
- Lag 2: 0.609
- Lag 4: 0.734
- Lag 6: 0.819
- Lag 8: 0.857

The strongest correlation was observed at **lag 8 weeks (0.857)**. This suggests that historical temperature data could be useful for predicting reservoir filling levels.

However, this relationship may partly reflect seasonal trends or autocorrelation and does not establish causality. Further analysis across other zones and with precipitation data is needed.

Also, when you first open Streamlit, as you navigate through the pages, you'll see progress indicators for each function as it loads.

# Links
- GitHub repository: https://github.com/xoksy/jouyit_IND320 
- streamlit app: https://jouyitind320-fafhfv6wkn6ecfeejafjgj.streamlit.app/
- video: https://youtu.be/06mNjvnv_Dk 

In [1]:
import requests
import pandas as pd
from io import StringIO


def download_currency_rates(start_date, end_date):

    # API endpoint
    url = "https://data.norges-bank.no/api/data/EXR/B.EUR+USD.NOK.SP"

    # Request parameters
    params = {
        "format": "csvfile",
        "startPeriod": start_date,
        "endPeriod": end_date,
        "locale": "en"
    }

    # Send API request
    response = requests.get(url, params=params, timeout=30)

    # Check for HTTP errors
    response.raise_for_status()

    # Convert CSV response into a pandas DataFrame
    df = pd.read_csv(StringIO(response.text))

    return df

In [2]:
currency_df = download_currency_rates(
    start_date="2024-01-01",
    end_date="2025-12-31"
)

# Display the first rows
display(currency_df.head())

# Display dataset information
print("Number of rows:", len(currency_df))
print("Columns:", currency_df.columns.tolist())
# Check which currencies are present
print(currency_df["BASE_CUR"].unique())
# Count observations for each currency
print(currency_df["BASE_CUR"].value_counts())

,FREQ,BASE_CUR,QUOTE_CUR,TENOR,DECIMALS,CALCULATED,UNIT_MULT,COLLECTION,TIME_PERIOD,OBS_VALUE
0,B,USD,NOK,SP,4,False,0,C,2024-01-02,10.2971
1,B,USD,NOK,SP,4,False,0,C,2024-01-03,10.3672
2,B,USD,NOK,SP,4,False,0,C,2024-01-04,10.3027
3,B,USD,NOK,SP,4,False,0,C,2024-01-05,10.3553
4,B,USD,NOK,SP,4,False,0,C,2024-01-08,10.3800


Number of rows: 1004
Columns: ['FREQ', 'BASE_CUR', 'QUOTE_CUR', 'TENOR', 'DECIMALS', 'CALCULATED', 'UNIT_MULT', 'COLLECTION', 'TIME_PERIOD', 'OBS_VALUE']
<ArrowStringArray>
['USD', 'EUR']
Length: 2, dtype: str
BASE_CUR
USD    502
EUR    502
Name: count, dtype: int64


In [3]:
# Keep only relevant columns
currency_df = currency_df[
    ["TIME_PERIOD", "BASE_CUR", "OBS_VALUE"]
].copy()

# Rename columns
currency_df.columns = ["date", "currency", "exchange_rate"]

# Convert date to datetime
currency_df["date"] = pd.to_datetime(currency_df["date"])

# Display cleaned data
display(currency_df.head())

#check timestamp for currency data
print("Date dtype:", currency_df["date"].dtype)

,date,currency,exchange_rate
0,2024-01-02,USD,10.2971
1,2024-01-03,USD,10.3672
2,2024-01-04,USD,10.3027
3,2024-01-05,USD,10.3553
4,2024-01-08,USD,10.3800


Date dtype: datetime64[us]


In [4]:
#CHECK TIMESTAMPS FOR ENTSOE DATA
import os
from pyspark.sql import SparkSession

os.environ["HADOOP_HOME"] = "C:/Hadoop/hadoop-3.3.1"
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

spark = (
    SparkSession.builder
    .appName("ENTSOE_CrossBorder_Flows")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"
    )
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

print("Spark version :", spark.version)

spark_connection = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(keyspace="energy_data", table="crossborder_flows") \
    .load()

# Display Spark session timezone
print(
    "Spark session timezone:",
    spark.conf.get("spark.sql.session.timeZone")
)

Spark version : 3.5.1
Spark session timezone: UTC


In [5]:
#CHECK TIMESTAMPS FOR OPEN METEO DATA
import pandas as pd

# Load NO1 weather data
weather_df = pd.read_parquet("../data/weather_NO1.parquet")

# Inspect timestamp information
print("Timestamp dtype:", weather_df["timestamp"].dtype)

print("\nFirst 10 timestamps:")
print(weather_df["timestamp"].head(10))

Timestamp dtype: datetime64[ms, UTC]

First 10 timestamps:
0   2023-12-31 23:00:00+00:00
1   2024-01-01 00:00:00+00:00
2   2024-01-01 01:00:00+00:00
3   2024-01-01 02:00:00+00:00
4   2024-01-01 03:00:00+00:00
5   2024-01-01 04:00:00+00:00
6   2024-01-01 05:00:00+00:00
7   2024-01-01 06:00:00+00:00
8   2024-01-01 07:00:00+00:00
9   2024-01-01 08:00:00+00:00
Name: timestamp, dtype: datetime64[ms, UTC]


In [8]:
#CHECK TIMESTAMPS FOR MAGASINSTATISTIKK DATA
url = (
    "https://biapi.nve.no/magasinstatistikk/"
    "api/Magasinstatistikk/HentOffentligData"
)

# Send the request
response = requests.get(url, timeout=30)

# Check that the request was successful
response.raise_for_status()

# Convert the JSON response into a DataFrame
data = response.json()
df = pd.DataFrame(data)
reservoirs = df.rename(columns={
    "dato_Id": "date_id",
    "omrType": "area_type",
    "omrnr": "area_number",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "filling_degree",
    "kapasitet_TWh": "capacity_TWh",
    "fylling_TWh": "filling_TWh",
    "neste_Publiseringsdato": "next_publication_date",
    "fyllingsgrad_forrige_uke": "filling_degree_previous_week",
    "endring_fyllingsgrad": "change_in_filling_degree"
})

reservoirs['date_id'] = pd.to_datetime(reservoirs['date_id'])
reservoirs['next_publication_date'] = pd.to_datetime(reservoirs['next_publication_date'])
reservoirs.loc[reservoirs["next_publication_date"].dt.year == 1,"next_publication_date"] = pd.NaT
reservoirs = reservoirs.sort_values("date_id")

print(reservoirs["date_id"].dtype)
print(reservoirs["date_id"].head(10))

datetime64[us]
7118    1995-01-08
7718    1995-01-08
14898   1995-01-08
9946    1995-01-08
8286    1995-01-08
11262   1995-01-08
4415    1995-01-08
14356   1995-01-08
2980    1995-01-08
14497   1995-01-15
Name: date_id, dtype: datetime64[us]


We investigated the timezone conventions used by the four data sources.

- ENTSO-E: Timestamps are handled in UTC.

- Open-Meteo: Data is initially requested using the Europe/Oslo timezone, but timestamps are converted to UTC during the loading process. The stored Parquet files therefore use UTC.

- Magasinstatistikk (NVE): The date_id column contains weekly dates without timezone information.

- Norges Bank (NB): Exchange rates are provided as daily dates (TIME_PERIOD), without specific hours or timezone information.

Conclusion:

ENTSO-E and Open-Meteo both use UTC, making their timestamps directly comparable without additional conversion. For Magasinstatistikk and Norges Bank, the dates should be kept as provided since they do not contain explicit timezone information. We should avoid applying arbitrary timezone conversions to these sources.

In [10]:
# Display available regions
print("Available regions:")
print(
    reservoirs[["area_type", "area_number"]]
    .drop_duplicates()
    .sort_values(["area_type", "area_number"])
    .to_string(index=False)
)

# Display relevant columns
print("\nDataset information:")
print(reservoirs[[
    "date_id",
    "area_type",
    "area_number",
    "filling_degree"
]].head(10))

# Display dataset date range
print("\nDate range:")
print("Start:", reservoirs["date_id"].min())
print("End:", reservoirs["date_id"].max())

Available regions:
area_type  area_number
       EL            1
       EL            2
       EL            3
       EL            4
       EL            5
       NO            0
     VASS            1
     VASS            2
     VASS            3

Dataset information:
         date_id area_type  area_number  filling_degree
7118  1995-01-08        EL            2        0.630947
7718  1995-01-08        EL            1        0.617574
14898 1995-01-08      VASS            3        0.554572
9946  1995-01-08        NO            0        0.603830
8286  1995-01-08        EL            4        0.542160
11262 1995-01-08      VASS            2        0.608856
4415  1995-01-08        EL            3        0.602376
14356 1995-01-08      VASS            1        0.635440
2980  1995-01-08        EL            5        0.616448
14497 1995-01-15      VASS            1        0.604392

Date range:
Start: 1995-01-08 00:00:00
End: 2026-11-08 00:00:00


In [12]:
# Display all available columns
print(reservoirs.columns.tolist())

# Check missing values
display(
    reservoirs.isnull().sum().to_frame("Missing values")
)

# Display descriptive statistics
display(reservoirs.describe())

['date_id', 'area_type', 'area_number', 'iso_year', 'iso_week', 'filling_degree', 'capacity_TWh', 'filling_TWh', 'next_publication_date', 'filling_degree_previous_week', 'change_in_filling_degree']


,Missing values
date_id,0
area_type,0
area_number,0
iso_year,0
iso_week,0
filling_degree,0
capacity_TWh,0
filling_TWh,0
next_publication_date,11322
filling_degree_previous_week,0


,date_id,area_number,iso_year,iso_week,filling_degree,capacity_TWh,filling_TWh,next_publication_date,filling_degree_previous_week,change_in_filling_degree
count,14958,14958.000000,14958.000000,14958.000000,14958.000000,14958.000000,14958.000000,3636,14958.000000,14958.000000
mean,2010-12-08 12:00:00,2.333333,2010.430806,26.484958,0.618375,29.145860,18.122800,2023-01-08 01:21:23.168316,0.618358,0.000018
min,1995-01-08 00:00:00,0.000000,1995.000000,1.000000,0.051653,6.003264,0.310089,2019-02-27 13:00:00,0.051653,-0.148802
25%,2002-12-22 00:00:00,1.000000,2002.000000,13.000000,0.456770,17.390587,7.295327,2021-02-01 19:00:00,0.456770,-0.022279
50%,2010-12-08 12:00:00,2.000000,2010.000000,26.000000,0.647673,23.237715,14.476256,2023-01-08 01:00:00,0.647682,-0.006888
75%,2018-11-25 00:00:00,3.000000,2018.000000,39.000000,0.801128,34.043590,22.174044,2024-12-13 07:00:00,0.801112,0.016192
max,2026-11-08 00:00:00,5.000000,2026.000000,53.000000,1.020072,87.437580,83.878470,2026-11-18 13:00:00,1.020037,0.345632
std,NaN,1.490762,9.193957,15.016641,0.214355,22.739066,15.947773,NaN,0.214333,0.031233


In [13]:
# Select region NO1
region_data = reservoirs[
    (reservoirs["area_type"] == "EL") &
    (reservoirs["area_number"] == 1)
].copy()

# Keep useful variables
region_data = region_data[
    [
        "date_id",
        "filling_degree",
        "filling_degree_previous_week",
        "change_in_filling_degree",
        "capacity_TWh",
        "filling_TWh",
        "iso_week"
    ]
]

# Sort chronologically
region_data = region_data.sort_values("date_id")

# Remove missing values in relevant columns
region_data = region_data.dropna()

# Set date as index
region_data = region_data.set_index("date_id")

print("Number of observations:", len(region_data))
display(region_data.head())

Number of observations: 1662


,filling_degree,filling_degree_previous_week,change_in_filling_degree,capacity_TWh,filling_TWh,iso_week
date_id,,,,,,
1995-01-08,0.617574,0.583705,0.033869,6.003264,3.707457,1
1995-01-15,0.579205,0.617574,-0.038369,6.003264,3.477117,2
1995-01-22,0.546713,0.579205,-0.032491,6.003264,3.282063,3
1995-01-29,0.504545,0.546713,-0.042168,6.003264,3.028916,4
1995-02-05,0.465335,0.504545,-0.039210,6.003264,2.793528,5


In [40]:
import plotly.express as px

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=region_data.index,
    y=region_data["filling_degree"],
    mode="lines",
    name="Historical filling"
))

fig.update_layout(
    title="Historical Reservoir Filling - NO1",
    xaxis_title="Date",
    yaxis_title="Filling Degree",
    hovermode="x unified"
)

fig.show()

In [ ]:
# Lets make some tests with the model parameters and evaluation settings
DEFAULT_MODEL_PARAMS = {
    "p": 1,
    "d": 1,
    "q": 1,
    "m": 52,
    "sp": 1,
    "sd": 0,
    "sq": 0
}

# Evaluation settings
HISTORY_WEEKS = 104
FORECAST_HORIZON = 52

# Define the different forecast starting points
test_periods = [
    "2023-01-01",
    "2024-01-01",
    "2025-01-01",
    "2026-01-01"
]

for date in test_periods:

    forecast_start = pd.Timestamp(date)

    # Find the first observation on or after the forecast date
    test_start_idx = region_data.index.searchsorted(forecast_start)

    train_start_idx = test_start_idx - HISTORY_WEEKS

    print(
        f"Forecast starting date: {forecast_start.date()} | "
        f"Training observations available: {max(0, test_start_idx - train_start_idx)}"
    )

Forecast starting date: 2023-01-01 | Training observations available: 104
Forecast starting date: 2024-01-01 | Training observations available: 104
Forecast starting date: 2025-01-01 | Training observations available: 104
Forecast starting date: 2026-01-01 | Training observations available: 104


In [42]:
def generate_baseline_predictions(train_data, horizon):

    # Naive forecast
    last_value = float(train_data["filling_degree"].iloc[-1])

    naive_predictions = [last_value] * horizon

    # Seasonal naive forecast
    seasonal_predictions = (
        train_data["filling_degree"]
        .iloc[-52:]
        .astype(float)
        .tolist()
    )

    # Repeat the seasonal pattern if horizon > 52
    seasonal_predictions = [
        seasonal_predictions[i % 52]
        for i in range(horizon)
    ]

    return naive_predictions, seasonal_predictions

In [43]:
from river import time_series
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate_forecast(train_data, test_data, model_params):

    # Initialize SNARIMAX
    model = time_series.SNARIMAX(**model_params)

    # Train the model
    for value in train_data["filling_degree"]:
        model.learn_one(float(value))

    # Generate static predictions
    predictions = list(
        model.forecast(horizon=len(test_data))
    )

    # Generate baseline predictions
    naive_pred, seasonal_pred = generate_baseline_predictions(
        train_data,
        len(test_data)
    )

    # Actual values
    actual = test_data["filling_degree"].astype(float).values

    # Evaluate all models
    model_predictions = {
        "SNARIMAX": predictions,
        "Naive": naive_pred,
        "Seasonal Naive": seasonal_pred
    }

    results = []

    for model_name, pred in model_predictions.items():

        pred = np.asarray(pred, dtype=float)

        # Prevent unrealistic predictions
        pred = np.clip(pred, 0, 1)

        results.append({
            "Model": model_name,
            "MAE": mean_absolute_error(actual, pred),
            "RMSE": np.sqrt(mean_squared_error(actual, pred)),
            "R2": r2_score(actual, pred)
        })

    return pd.DataFrame(results), predictions

In [44]:
# Select forecast starting date
forecast_start = pd.Timestamp("2025-01-01")

# Find corresponding index
test_start_idx = region_data.index.searchsorted(forecast_start)

# Select training history
train_data = region_data.iloc[
    test_start_idx - HISTORY_WEEKS:test_start_idx
]

# Select test period
test_data = region_data.iloc[
    test_start_idx:test_start_idx + FORECAST_HORIZON
]

print("Training observations:", len(train_data))
print("Testing observations:", len(test_data))

print(
    "Training period:",
    train_data.index.min(),
    "to",
    train_data.index.max()
)

print(
    "Testing period:",
    test_data.index.min(),
    "to",
    test_data.index.max()
)

Training observations: 104
Testing observations: 52
Training period: 2023-01-08 00:00:00 to 2024-12-29 00:00:00
Testing period: 2025-01-05 00:00:00 to 2025-12-28 00:00:00


In [45]:
results, predictions = evaluate_forecast(
    train_data=train_data,
    test_data=test_data,
    model_params=DEFAULT_MODEL_PARAMS
)

display(results)

,Model,MAE,RMSE,R2
0,SNARIMAX,0.112489,0.128360,0.575621
1,Naive,0.179580,0.237866,-0.457348
2,Seasonal Naive,0.126983,0.144058,0.465473


In [46]:
fig = go.Figure()

# Historical training data
fig.add_trace(go.Scatter(
    x=train_data.index,
    y=train_data["filling_degree"],
    mode="lines",
    name="Training data"
))

# Actual test data
fig.add_trace(go.Scatter(
    x=test_data.index,
    y=test_data["filling_degree"],
    mode="lines",
    name="Actual filling"
))

# SNARIMAX predictions
fig.add_trace(go.Scatter(
    x=test_data.index,
    y=np.clip(predictions, 0, 1),
    mode="lines",
    name="SNARIMAX predictions",
    line=dict(dash="dash")
))

fig.update_layout(
    title="SNARIMAX - Forecasting NO1",
    xaxis_title="Date",
    yaxis_title="Filling Degree",
    hovermode="x unified"
)

fig.show()

In [ ]:
#test of different history lengths and forecast starting points
history_settings = {
    "2 years": 104,
    "3 years": 156,
    "5 years": 260,
    "7 years": 364,
    "10 years": 520
}

history_results = []

for history_name, history_weeks in history_settings.items():

    for year in [2023, 2024, 2025]:

        print(f"Testing {history_name} - {year}...")

        forecast_start = pd.Timestamp(f"{year}-01-01")

        test_start_idx = region_data.index.searchsorted(forecast_start)

        # Select historical data
        train_data = region_data.iloc[
            test_start_idx - history_weeks:test_start_idx
        ]

        # Select 52 weeks of test data
        test_data = region_data.iloc[
            test_start_idx:test_start_idx + 52
        ]

        if len(train_data) < history_weeks or len(test_data) < 52:
            continue

        # Evaluate models
        results, _ = evaluate_forecast(
            train_data,
            test_data,
            DEFAULT_MODEL_PARAMS
        )

        results["History"] = history_name
        results["Year"] = year

        history_results.append(results)

# Combine results
history_results_df = pd.concat(
    history_results,
    ignore_index=True
)

display(
    history_results_df.sort_values(["Year", "History", "MAE"])
)

Testing 2 years - 2023...
Testing 2 years - 2024...
Testing 2 years - 2025...
Testing 3 years - 2023...
Testing 3 years - 2024...
Testing 3 years - 2025...
Testing 5 years - 2023...
Testing 5 years - 2024...
Testing 5 years - 2025...
Testing 7 years - 2023...
Testing 7 years - 2024...
Testing 7 years - 2025...
Testing 10 years - 2023...
Testing 10 years - 2024...
Testing 10 years - 2025...


,Model,MAE,RMSE,R2,History,Year
38,Seasonal Naive,0.130711,0.158567,0.698985,10 years,2023
36,SNARIMAX,0.131869,0.149856,0.731146,10 years,2023
37,Naive,0.247045,0.291397,-0.016565,10 years,2023
2,Seasonal Naive,0.130711,0.158567,0.698985,2 years,2023
0,SNARIMAX,0.189081,0.217133,0.435559,2 years,2023
1,Naive,0.247045,0.291397,-0.016565,2 years,2023
11,Seasonal Naive,0.130711,0.158567,0.698985,3 years,2023
9,SNARIMAX,0.182456,0.212536,0.459207,3 years,2023
10,Naive,0.247045,0.291397,-0.016565,3 years,2023
20,Seasonal Naive,0.130711,0.158567,0.698985,5 years,2023


In [50]:
# Average performance across the three evaluation years
history_summary = (
    history_results_df
    .groupby(["History", "Model"])[["MAE", "RMSE", "R2"]]
    .mean()
    .reset_index()
)

display(
    history_summary.sort_values(["Model", "MAE"])
)

,History,Model,MAE,RMSE,R2
0,10 years,Naive,0.231017,0.272071,-0.163420
3,2 years,Naive,0.231017,0.272071,-0.163420
6,3 years,Naive,0.231017,0.272071,-0.163420
9,5 years,Naive,0.231017,0.272071,-0.163420
12,7 years,Naive,0.231017,0.272071,-0.163420
13,7 years,SNARIMAX,0.172841,0.196767,0.364206
1,10 years,SNARIMAX,0.176198,0.200938,0.332416
10,5 years,SNARIMAX,0.182607,0.207396,0.285120
7,3 years,SNARIMAX,0.212508,0.243485,0.047205
4,2 years,SNARIMAX,0.212542,0.240705,0.086785


In [51]:
snarimax_history = history_summary[
    history_summary["Model"] == "SNARIMAX"
].copy()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=snarimax_history["History"],
    y=snarimax_history["MAE"],
    mode="lines+markers",
    name="SNARIMAX MAE"
))

fig.update_layout(
    title="SNARIMAX Performance vs Training History",
    xaxis_title="Training History",
    yaxis_title="Mean Absolute Error"
)

fig.show()

In [ ]:
#test of different model configurations
model_configurations = {
    "A - Current": {
        "p": 1, "d": 1, "q": 1,
        "m": 52, "sp": 1, "sd": 0, "sq": 0
    },

    "B - Simple": {
        "p": 1, "d": 1, "q": 0,
        "m": 52, "sp": 0, "sd": 0, "sq": 0
    },

    "C - Seasonal": {
        "p": 1, "d": 1, "q": 1,
        "m": 52, "sp": 1, "sd": 1, "sq": 0
    },

    "D - Alternative": {
        "p": 2, "d": 1, "q": 1,
        "m": 52, "sp": 1, "sd": 0, "sq": 0
    }
}

configuration_results = []

for config_name, params in model_configurations.items():

    for year in [2023, 2024, 2025]:

        print(f"Testing {config_name} - {year}...")

        forecast_start = pd.Timestamp(f"{year}-01-01")

        test_start_idx = region_data.index.searchsorted(forecast_start)

        train_data = region_data.iloc[
            test_start_idx - 7 * 52:test_start_idx
        ]

        test_data = region_data.iloc[
            test_start_idx:test_start_idx + 52
        ]

        if len(train_data) < 7 * 52 or len(test_data) < 52:
            continue

        results, _ = evaluate_forecast(
            train_data,
            test_data,
            params
        )

        # Keep only SNARIMAX results
        snarimax_result = results[
            results["Model"] == "SNARIMAX"
        ].copy()

        snarimax_result["Configuration"] = config_name
        snarimax_result["Year"] = year

        configuration_results.append(snarimax_result)

configuration_results_df = pd.concat(
    configuration_results,
    ignore_index=True
)

display(
    configuration_results_df.sort_values(["Year", "MAE"])
)

configuration_summary = (
    configuration_results_df
    .groupby("Configuration")[["MAE", "RMSE", "R2"]]
    .mean()
    .reset_index()
    .sort_values("MAE")
)

display(configuration_summary)

Testing A - Current - 2023...
Testing A - Current - 2024...
Testing A - Current - 2025...
Testing B - Simple - 2023...
Testing B - Simple - 2024...
Testing B - Simple - 2025...
Testing C - Seasonal - 2023...
Testing C - Seasonal - 2024...
Testing C - Seasonal - 2025...
Testing D - Alternative - 2023...
Testing D - Alternative - 2024...
Testing D - Alternative - 2025...


,Model,MAE,RMSE,R2,Configuration,Year
6,SNARIMAX,0.103667,0.137543,0.773514,C - Seasonal,2023
0,SNARIMAX,0.131336,0.149184,0.733555,A - Current,2023
9,SNARIMAX,0.142451,0.165557,0.671858,D - Alternative,2023
3,SNARIMAX,0.336168,0.376406,-0.696207,B - Simple,2023
7,SNARIMAX,0.080174,0.102690,0.869838,C - Seasonal,2024
10,SNARIMAX,0.249730,0.276146,0.058748,D - Alternative,2024
1,SNARIMAX,0.281763,0.319480,-0.259844,A - Current,2024
4,SNARIMAX,0.363831,0.393168,-0.908035,B - Simple,2024
2,SNARIMAX,0.105425,0.121637,0.618906,A - Current,2025
11,SNARIMAX,0.115449,0.137882,0.510321,D - Alternative,2025


,Configuration,MAE,RMSE,R2
2,C - Seasonal,0.102228,0.134428,0.652863
3,D - Alternative,0.169210,0.193195,0.413642
0,A - Current,0.172841,0.196767,0.364206
1,B - Simple,0.303265,0.335870,-0.687892


In [ ]:
#test of different history lengths with the best model configuration
seasonal_history_results = []

history_settings = {
    "2 years": 104,
    "3 years": 156,
    "5 years": 260,
    "7 years": 364,
    "10 years": 520
}

seasonal_params = model_configurations["C - Seasonal"]

for history_name, history_weeks in history_settings.items():

    for year in [2023, 2024, 2025]:

        print(f"Testing {history_name} - {year}...")

        forecast_start = pd.Timestamp(f"{year}-01-01")

        test_start_idx = region_data.index.searchsorted(forecast_start)

        train_data = region_data.iloc[
            test_start_idx - history_weeks:test_start_idx
        ]

        test_data = region_data.iloc[
            test_start_idx:test_start_idx + 52
        ]

        if len(train_data) < history_weeks or len(test_data) < 52:
            continue

        results, _ = evaluate_forecast(
            train_data,
            test_data,
            seasonal_params
        )

        snarimax_result = results[
            results["Model"] == "SNARIMAX"
        ].copy()

        snarimax_result["History"] = history_name
        snarimax_result["Year"] = year

        seasonal_history_results.append(snarimax_result)

seasonal_history_df = pd.concat(
    seasonal_history_results,
    ignore_index=True
)

display(
    seasonal_history_df.sort_values(["Year", "MAE"])
)

Testing 2 years - 2023...
Testing 2 years - 2024...
Testing 2 years - 2025...
Testing 3 years - 2023...
Testing 3 years - 2024...
Testing 3 years - 2025...
Testing 5 years - 2023...
Testing 5 years - 2024...
Testing 5 years - 2025...
Testing 7 years - 2023...
Testing 7 years - 2024...
Testing 7 years - 2025...
Testing 10 years - 2023...
Testing 10 years - 2024...
Testing 10 years - 2025...


,Model,MAE,RMSE,R2,History,Year
12,SNARIMAX,0.088489,0.123077,0.818648,10 years,2023
0,SNARIMAX,0.089854,0.119203,0.829888,2 years,2023
3,SNARIMAX,0.091075,0.131768,0.792135,3 years,2023
9,SNARIMAX,0.103667,0.137543,0.773514,7 years,2023
6,SNARIMAX,0.120310,0.152198,0.722680,5 years,2023
4,SNARIMAX,0.058398,0.071343,0.937175,3 years,2024
7,SNARIMAX,0.073289,0.096059,0.886104,5 years,2024
10,SNARIMAX,0.080174,0.102690,0.869838,7 years,2024
13,SNARIMAX,0.082148,0.104678,0.864750,10 years,2024
1,SNARIMAX,0.139640,0.168574,0.649241,2 years,2024


In [ ]:
#test of different forecast horizons with the best model configuration
horizon_settings = {
    "1 month": 4,
    "3 months": 13,
    "6 months": 26,
    "12 months": 52
}

HISTORY_WEEKS = 3 * 52

horizon_results = []

for year in [2023, 2024, 2025]:

    print(f"Testing year {year}...")

    forecast_start = pd.Timestamp(f"{year}-01-01")

    test_start_idx = region_data.index.searchsorted(forecast_start)

    train_data = region_data.iloc[
        test_start_idx - HISTORY_WEEKS:test_start_idx
    ]

    test_data = region_data.iloc[
        test_start_idx:test_start_idx + 52
    ]

    if len(train_data) < HISTORY_WEEKS or len(test_data) < 52:
        continue

    # Initialize and train SNARIMAX
    model = time_series.SNARIMAX(**seasonal_params)

    for value in train_data["filling_degree"]:
        model.learn_one(float(value))

    # Generate a single 52-week static forecast
    predictions = np.array(
        list(model.forecast(horizon=52)),
        dtype=float
    )

    predictions = np.clip(predictions, 0, 1)

    # Evaluate each horizon
    for horizon_name, horizon in horizon_settings.items():

        actual = test_data["filling_degree"].iloc[:horizon].values
        predicted = predictions[:horizon]

        horizon_results.append({
            "Year": year,
            "Horizon": horizon_name,
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "R2": r2_score(actual, predicted)
        })

horizon_results_df = pd.DataFrame(horizon_results)

display(
    horizon_results_df.sort_values(["Horizon", "Year"])
)

Testing year 2023...
Testing year 2024...
Testing year 2025...


,Year,Horizon,MAE,RMSE,R2
0,2023,1 month,0.023378,0.026638,0.436999
4,2024,1 month,0.027439,0.030045,0.577575
8,2025,1 month,0.012887,0.015325,0.851011
3,2023,12 months,0.091075,0.131768,0.792135
7,2024,12 months,0.058398,0.071343,0.937175
11,2025,12 months,0.107687,0.149725,0.422586
1,2023,3 months,0.018992,0.021195,0.979419
5,2024,3 months,0.041728,0.043718,0.890630
9,2025,3 months,0.011321,0.013301,0.992150
2,2023,6 months,0.039942,0.060985,0.911095


In [ ]:
#here is the function with the best default parameters for the SNARIMAX model and the evaluation settings
#the explanation is in the description of the notebook 
import pandas as pd
import numpy as np

from river import time_series


def forecast_reservoir_filling(
    reservoirs,
    area_type="EL",
    area_number=1,
    history_years=3,
    horizon_weeks=13,
    p=1,
    d=1,
    q=1,
    sp=1,
    sd=1,
    sq=0,
    m=52,
    data_source="Magasinstatistikk"
):

    # 1. Check selected data source
    if data_source != "Magasinstatistikk":
        raise ValueError("Only Magasinstatistikk is currently supported.")
    
    if history_years is not None and history_years <= 0:
        raise ValueError("history_years must be positive.")

    if horizon_weeks <= 0:
        raise ValueError("horizon_weeks must be positive.")

    if m <= 0:
        raise ValueError("m must be positive.")

    # 2. Select the requested region
    region_data = reservoirs[
        (reservoirs["area_type"] == area_type) &
        (reservoirs["area_number"] == area_number)
    ][["date_id", "filling_degree"]].copy()

    if region_data.empty:
        raise ValueError("No data available for the selected region.")

    # 3. Prepare the time series
    region_data["date_id"] = pd.to_datetime(region_data["date_id"])

    region_data = (
        region_data
        .sort_values("date_id")
        .drop_duplicates(subset="date_id")
        .set_index("date_id")
    )

    region_data["filling_degree"] = pd.to_numeric(
        region_data["filling_degree"],
        errors="coerce"
    )

    region_data = region_data.dropna()

    # 4. Select the historical period
    end_date = region_data.index.max()

    if history_years is not None:
        start_date = end_date - pd.DateOffset(years=history_years)

        region_data = region_data[
            region_data.index >= start_date
        ]

    # 5. Check data availability
    if len(region_data) < 2 * m:
        raise ValueError(
            "Not enough historical observations for SNARIMAX."
        )

    # Check weekly frequency
    intervals = region_data.index.to_series().diff().dropna()

    if not (intervals == pd.Timedelta(days=7)).all():
        raise ValueError(
            "The selected region does not have a regular weekly frequency."
        )

    # 6. Initialize SNARIMAX
    model = time_series.SNARIMAX(
        p=p,
        d=d,
        q=q,
        sp=sp,
        sd=sd,
        sq=sq,
        m=m
    )

    # 7. Train the model
    for value in region_data["filling_degree"]:
        model.learn_one(float(value))

    # 8. Generate future predictions
    predictions = list(
        model.forecast(horizon=horizon_weeks)
    )
    
    if len(predictions) != horizon_weeks:
        raise ValueError("The model did not return the expected number of predictions.")

    if not np.isfinite(predictions).all():
        raise ValueError("The model produced invalid predictions.")
        
    predictions = np.clip(predictions, 0, 1)

    # 9. Generate future dates
    last_date = region_data.index.max()

    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=7),
        periods=horizon_weeks,
        freq="7D"
    )

    # 10. Create the forecast DataFrame
    forecast_df = pd.DataFrame({
        "date_id": future_dates,
        "predicted_filling_degree": predictions
    })

    return forecast_df

In [66]:
forecast = forecast_reservoir_filling(
    reservoirs=reservoirs
)

display(forecast)

,date_id,predicted_filling_degree
0,2026-11-15,0.766912
1,2026-11-22,0.723612
2,2026-11-29,0.689215
3,2026-12-06,0.661320
4,2026-12-13,0.635202
5,2026-12-20,0.625095
6,2026-12-27,0.587389
7,2027-01-03,0.545457
8,2027-01-10,0.493539
9,2027-01-17,0.444044


In [68]:
import plotly.graph_objects as go

# Select the same region as the forecast
historical_data = reservoirs[
    (reservoirs["area_type"] == "EL") &
    (reservoirs["area_number"] == 1)
].copy()

historical_data["date_id"] = pd.to_datetime(historical_data["date_id"])

historical_data = historical_data.sort_values("date_id")

# Keep the last 3 years
start_date = historical_data["date_id"].max() - pd.DateOffset(years=3)

historical_data = historical_data[
    historical_data["date_id"] >= start_date
]

# Create figure
fig = go.Figure()

# Historical data
fig.add_trace(go.Scatter(
    x=historical_data["date_id"],
    y=historical_data["filling_degree"] * 100,
    mode="lines",
    name="Historical filling degree",
    line=dict(width=2),
    hovertemplate="%{x|%d/%m/%Y}<br>Filling: %{y:.2f}%<extra></extra>"
))

# Forecast
fig.add_trace(go.Scatter(
    x=forecast["date_id"],
    y=forecast["predicted_filling_degree"] * 100,
    mode="lines+markers",
    name="Forecast",
    line=dict(dash="dash", width=2),
    marker=dict(size=7),
    hovertemplate="%{x|%d/%m/%Y}<br>Predicted filling: %{y:.2f}%<extra></extra>"
))

# Forecast starting point
fig.add_vline(
    x=forecast["date_id"].iloc[0],
    line_dash="dot",
    annotation_text="Forecast start",
    annotation_position="top"
)

# Layout
fig.update_layout(
    title="Reservoir Filling Degree Forecast – EL1 (NO1)",
    xaxis_title="Date",
    yaxis_title="Filling degree (%)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.update_yaxes(range=[0, 100])

fig.show()